In [ ]:
# devices_with_structures.py
# -------------------------------------------------
# Build structures once, pull one or all device tables,
# join on PoleID == GISID, and return a dict of DataFrames
# plus a combined DataFrame.

from __future__ import annotations
import pyodbc
import pandas as pd
from typing import Dict, Tuple, Optional, List
from dbfread import DBF
from pathlib import Path
from simpledbf import Dbf5
# -----------------------------
# Connection helpers
# -----------------------------

def get_connection(server: str, database: str):
    conn_str = (
        "Driver={SQL Server};"
        f"Server={server};"
        f"Database={database};"
        "Trusted_Connection=yes;"
    )
    return pyodbc.connect(conn_str)

def run_query_trusted(query: str, server: str, database: str) -> pd.DataFrame:
    with get_connection(server, database) as conn:
        return pd.read_sql_query(query, conn)

def run_query_with_temp_trusted(query_setup: str, query_final: str, server: str, database: str) -> pd.DataFrame:
    conn = get_connection(server, database)
    cur = conn.cursor()
    cur.execute("SET NOCOUNT ON; " + query_setup)
    while cur.nextset():
        pass
    df = pd.read_sql_query(query_final, conn)
    cur.close()
    conn.close()
    return df

# -----------------------------
# Reference maps
# -----------------------------

DIV_CODE_MAP = {
    13: 'Canandaigua', 14: 'Central/Rochester', 15: 'Genesee Valley/Filmore',
    18: 'Lakeshore/Sodus', 26: 'Brewster', 30: 'Mechanicville',
    34: 'Plattsburgh', 38: 'Oneonta', 48: 'Liberty', 51: 'Lancaster',
    55: 'Lockport', 57: 'Elmira', 61: 'Hornell', 65: 'Geneva',
    68: 'Ithaca', 70: 'Auburn', 71: 'Binghamton', 79: 'Brunswick',
    80: 'Penobscot/Rockland', 81: 'Alfred', 82: 'Portland', 83: 'Augusta',
    84: 'Fairfield', 85: 'Lewiston', 86: 'Farmington', 87: 'Dover',
    88: 'Skowhegan', 89: 'Bridgton'
}

OPCO_MAP = {
    9301: 'NYSEG',
    9302: 'RGE',
    9310: 'CMP'
}

# -----------------------------
# Device specifications & SQL
# -----------------------------

DEVICE_SPECS: Dict[str, Dict[str, str]] = {
    "transformer": {
        "unit_table": "Electric.XfmrUnit_evw",
        "parent_table": "Electric.Transformer_evw",
        "unit_id": "TransformerGUID",
        "parent_id": "GlobalID"
    },
    "recloser": {
        "unit_table": "Electric.RECLOSERUNIT_EVW",
        "parent_table": "Electric.DYNAMICPROTECTIVEDEVICE_EVW",
        "unit_id": "RecloserGUID",
        "parent_id": "GlobalID"
    },
    "sectionalizer": {
        "unit_table": "Electric.SECTIONALIZERUNIT",
        "parent_table": "Electric.DYNAMICPROTECTIVEDEVICE_EVW",
        "unit_id": "SectionalizerGUID",
        "parent_id": "GlobalID"
    },
    "line_sensor": {
        "unit_table": "Electric.LineSensorUnit_evw",
        "parent_table": "Electric.DYNAMICPROTECTIVEDEVICE_EVW",
        "unit_id": "SensorGUID",
        "parent_id": "GlobalID"
    },
    "capacitor": {
        "unit_table": "Electric.CapacitorUnit_evw",
        "parent_table": "Electric.PFCorrectingEquipment_evw",
        "unit_id": "CapacitorBankGUID",
        "parent_id": "GlobalID"
    },
    "voltage_regulator": {
        "unit_table": "Electric.VOLTAGEREGULATORUNIT_EVW",
        "parent_table": "Electric.VOLTAGEREGULATOR_EVW",
        "unit_id": "VoltageRegulatorGUID",
        "parent_id": "GlobalID"
    },
    "fuse_cutout": {
        "unit_table": "Electric.FUSEUNIT_EVW",
        "parent_table": "Electric.Fuse_evw",
        "unit_id": "FuseObjectGUID",
        "parent_id": "GlobalID"
    },
    "switch": {
        "unit_table": "Electric.SWITCHUNIT_EVW",
        "parent_table": "Electric.SWITCH_EVW",
        "unit_id": "SwitchGUID",
        "parent_id": "GlobalID"
    },
    "fault_indicator": {
        "unit_table": "Electric.FaultIndUnit_evw",
        "parent_table": "Electric.MISCNETWORKFEATURE_EVW",
        "unit_id": "MiscObjectGUID",
        "parent_id": "GlobalID"
    },
    "terminating_cabinet": {
        "unit_table": "Electric.TermCabinetUnit_evw",
        "parent_table": "Electric.MISCNETWORKFEATURE_EVW",
        "unit_id": "MiscObjectGUID",
        "parent_id": "GlobalID"
    },
    "switchgear": {
        "unit_table": "Electric.SwitchGearUnit_evw",
        "parent_table": "Electric.MISCNETWORKFEATURE_EVW",
        "unit_id": "MiscObjectGUID",
        "parent_id": "GlobalID"
    },
    "net_protector": {
        "unit_table": "Electric.NetProtectorUnit_evw",
        "parent_table": "Electric.MISCNETWORKFEATURE_EVW",
        "unit_id": "MiscObjectGUID",
        "parent_id": "GlobalID"
    }
}

DEVICE_SQL_TEMPLATE = """
SELECT 
    ru.[GISID], ru.[SAPID], ru.[CUDesc], ru.[SerialNumber], ru.[WorkOrderID], 
    ru.[InSvcDate], ru.[{unit_id}] AS DeviceGUID, ru.[ConstructionStatus], 
    dpd.[OperatingVoltage], dpd.[PoleID]
FROM {unit_table} ru
LEFT JOIN {parent_table} dpd
    ON ru.[{unit_id}] = dpd.[{parent_id}];
"""

# -----------------------------
# Structures query (with X/Y)
# -----------------------------

STRUCTURES_QUERY_SETUP = """
DROP TABLE IF EXISTS #Structure;

SELECT [COCode], [SAPID], [GISID], [DivCode], [FuncLoc],
 TrueX AS Location_X, TrueY AS Location_Y,
 'Pole' AS Type
INTO #Structure
FROM Electric.SupportStructure_evw

UNION ALL

SELECT [COCode], [SAPID], [GISID], [DivCode], [FuncLoc],
 TrueX AS Location_X, TrueY AS Location_Y,
 CASE 
 WHEN [SubtypeCD] = 1 THEN 'Handhole'
 WHEN [SubtypeCD] = 2 THEN 'Mandhole'
 WHEN [SubtypeCD] = 3 THEN 'Vault'
 ELSE 'NA_Underground'
 END AS Type
FROM Electric.UndergroundStructure_evw

UNION ALL

SELECT [COCode], [SAPID], [GISID], [DivCode], [FuncLoc],
 TrueX AS Location_X, TrueY AS Location_Y,
 'Vault Pad' AS Type
FROM Electric.SURFACESTRUCTURE_EVW

UNION ALL

SELECT [COCode], [SAPID], [GISID], NULL AS [DivCode], [FuncLoc],
 NULL AS Location_X, NULL AS Location_Y,
 'Vert' AS Type
FROM Electric.Vert;
"""

STRUCTURES_QUERY_FINAL = "SELECT * FROM #Structure;"


def get_most_recent_folder(base_path):
    base = Path(base_path)
    
    # List only directories and exclude those ending with '_report'
    dirs = [d for d in base.iterdir() if d.is_dir() and not d.name.lower().endswith('_report')]
    
    if not dirs:
        return None
    
    # Get the most recently created directory
    most_recent = max(dirs, key=lambda d: d.stat().st_ctime)
    return most_recent



# -----------------------------
# Public API
# -----------------------------

def get_devices_with_structures(
    server: str,
    database: str,
    device: Optional[str] = None,
    save_to_csv: Optional[str] = None,
    csv_path: Optional[str] = None
) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Fetch one or all device tables, join with structures on PoleID==GISID, and return:
    - dict of dataframes per device
    - combined dataframe of all devices
    """

    all_dfs = []
    if device is None:
        
        # Example usage
        base_dir = r"\\clornas01\GIS\NY-Distribution\NY_Archive"
        latest_folder = get_most_recent_folder(base_dir)
        print(f"Most recently created folder: {latest_folder}")

        # Find rstru.dbf anywhere under latest_folder
        dbf_path = next((p for p in Path(latest_folder).rglob("rstru.dbf")), None)

        if dbf_path is None:
            raise FileNotFoundError("rstru.dbf not found inside the latest folder.")

        # Derive CSV path from the DBF path (avoids None)
        pickle_path = dbf_path.with_suffix(".pkl")

        if pickle_path.exists():
            df_coord_structure = pd.read_pickle(pickle_path)
        else:
            # Read DBF then write CSV next to it
            try:
                dbf = Dbf5(str(dbf_path), codec="cp1252")
                df_coord_structure = dbf.to_dataframe()
            except UnicodeDecodeError:
                dbf = Dbf5(str(dbf_path), codec="latin-1")
                df_coord_structure = dbf.to_dataframe()
            
            df_coord_structure.to_pickle(pickle_path)
        df_structures = run_query_with_temp_trusted(query_setup=STRUCTURES_QUERY_SETUP, query_final=STRUCTURES_QUERY_FINAL, server=server, database=database)
        for device, spec in DEVICE_SPECS.items():
            sql = DEVICE_SQL_TEMPLATE.format(**spec)
            df = run_query_trusted(query=sql, server=server, database=database)
        
            # Store individual DataFrame if needed
            globals()[f"df_{device}"] = df
            df['Device'] = f'{device}'
            # Append to list
            all_dfs.append(df)
        df_all = pd.concat(all_dfs, ignore_index=True)
        df_all = df_all.merge(
        df_structures[['GISID', 'COCode', 'DivCode', 'FuncLoc', 'Type', 'Location_X', 'Location_Y']],
        left_on='PoleID',
        right_on='GISID',
        how='left'
        ).rename(columns={'GISID_x': 'GISID', 'Type':'StructureType'}).drop(columns=['GISID_y']).reset_index(drop=True)
        df_coord_structure = df_coord_structure[['X', 'Y', 'FUNCLOC']]
        df_all = df_all.merge(df_coord_structure, left_on='FuncLoc', right_on='FUNCLOC', how='left').drop_duplicates(subset=['SAPID', 'FUNCLOC']).reset_index(drop=True)
        df_all['Location_X'] = df_all['X'].combine_first(df_all['Location_X'])
        df_all['Location_Y'] = df_all['Y'].combine_first(df_all['Location_Y'])
        df_all= df_all.drop(columns=['X', 'Y', 'FUNCLOC']).reset_index(drop=True)
        
        if save_to_csv and csv_path:
            try:
                df_all.to_csv(csv_path, index=False)
                print(f"Saved to CSV at: {csv_path}")
            except Exception as e:
                print(f"Failed to save CSV: {e}")

        return df_all
    else:
        base_dir = r"\\clornas01\GIS\NY-Distribution\NY_Archive"
        latest_folder = get_most_recent_folder(base_dir)
        print(f"Most recently created folder: {latest_folder}")

        # Find rstru.dbf anywhere under latest_folder
        dbf_path = next((p for p in Path(latest_folder).rglob("rstru.dbf")), None)

        if dbf_path is None:
            raise FileNotFoundError("rstru.dbf not found inside the latest folder.")

        # Derive CSV path from the DBF path (avoids None)
        pickle_path = dbf_path.with_suffix(".pkl")
        if pickle_path.exists():
            df_coord_structure = pd.read_pickle(pickle_path)
        else:
            # Read DBF then write CSV next to it
            try:
                dbf = Dbf5(str(dbf_path), codec="cp1252")
                df_coord_structure = dbf.to_dataframe()
            except UnicodeDecodeError:
                dbf = Dbf5(str(dbf_path), codec="latin-1")
                df_coord_structure = dbf.to_dataframe()
            
            df_coord_structure.to_pickle(pickle_path)
        
        
        device = device.lower()
        df_structures = run_query_with_temp_trusted(query_setup=STRUCTURES_QUERY_SETUP, query_final=STRUCTURES_QUERY_FINAL, server=server, database=database)

        if device in DEVICE_SPECS:
            spec = DEVICE_SPECS[device]
            sql = DEVICE_SQL_TEMPLATE.format(**spec)

            df = run_query_trusted(query=sql, server=server, database=database)

            df = df.merge(
                df_structures[['GISID', 'COCode', 'DivCode', 'FuncLoc', 'Type', 'Location_X', 'Location_Y']],
                left_on='PoleID',
                right_on='GISID',
                how='left'
            ).rename(columns={'GISID_x': 'GISID', 'Type':'StructureType'}).drop(columns=['GISID_y']).reset_index(drop=True)

            df['Device'] = device
            df_coord_structure = df_coord_structure[['X', 'Y', 'FUNCLOC']]
            df = df.merge(df_coord_structure, left_on='FuncLoc', right_on='FUNCLOC', how='left').drop_duplicates(subset=['SAPID', 'FUNCLOC']).reset_index(drop=True)
            df['Location_X'] = df['X'].combine_first(df['Location_X'])
            df['Location_Y'] = df['Y'].combine_first(df['Location_Y'])
            df= df.drop(columns=['X', 'Y', 'FUNCLOC']).reset_index(drop=True)
            if save_to_csv and csv_path:
                try:
                    df.to_csv(csv_path, index=False)
                    print(f"Saved to CSV at: {csv_path}")
                except Exception as e:
                    print(f"Failed to save CSV: {e}")
            return df

        else:
            print(f"Device '{device}' not found. Available options: {list(DEVICE_SPECS.keys())}")
            return None, None

# -----------------------------
# Optional: list available devices
# -----------------------------

def list_devices() -> List[str]:
    return list(DEVICE_SPECS.keys())

In [2]:
get_devices_with_structures(server='nygisprd19', database='NYGIS', device='transformer')

Most recently created folder: \\clornas01\GIS\NY-Distribution\NY_Archive\2025-10-22-5


UnboundLocalError: local variable 'Dbf5' referenced before assignment